# Tech-Triathlon 2026 — Waypoint Group Datathon

This notebook documents the reproducible final solution for Task 1, Task 2A, and Task 2B. The exact prediction metric is not stated in the official booklet, so the workflow reports multiple transparent chronological metrics and does not describe any surrogate as official.

All predictive models were trained from scratch on the supplied synthetic data. No proprietary modelling API, pretrained predictive model, external dataset, AutoML platform, or low-code/no-code modeller was used.

In [1]:
from pathlib import Path
import json, os, sys, warnings
import joblib
import numpy as np
import pandas as pd

os.environ["LOKY_MAX_CPU_COUNT"] = "16"
warnings.filterwarnings("ignore", message=".*mismatched devices.*")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

WORK = Path.cwd().resolve()
if not ((WORK / "src").is_dir() and (WORK / "models").is_dir()):
    candidates = [p / "work" for p in [WORK, *WORK.parents] if (p / "work" / "src").exists()]
    if not candidates:
        raise FileNotFoundError("Run this notebook from the extracted submission folder or project work directory")
    WORK = candidates[0]
data_candidates = [p / "Data" for p in [WORK, *WORK.parents] if (p / "Data").is_dir()]
if not data_candidates:
    raise FileNotFoundError("Place the supplied Data folder beside the submission folder")
DATA = data_candidates[0]
PROJECT = DATA.parent
sys.path.insert(0, str(WORK / "src"))

print("Project:", PROJECT)
print("Data:", DATA)
print("Python:", sys.version.split()[0])

Project: C:\Users\Sandun\Downloads\TechTri\Tech-Triathlon_Datathon_Challenge
Data: C:\Users\Sandun\Downloads\TechTri\Tech-Triathlon_Datathon_Challenge\Data
Python: 3.12.3


## Data inventory and integrity

In [2]:
audit = json.loads((WORK / "reports" / "initial_audit.json").read_text(encoding="utf-8"))
inventory = pd.DataFrame([
    {"file": item["path"], "rows": item["rows"], "columns": len(item["columns"]), "duplicate_rows": item["duplicate_rows"]}
    for item in audit["files"]
])
display(inventory)
print("Join audit:", json.dumps(audit["joins"], indent=2))
print("Template audit:", json.dumps(audit["templates"], indent=2))

                                           file  rows  columns  duplicate_rows
                 Data\General Data\calendar.csv   910       12               0
          Data\General Data\district_travel.csv    12        8               0
                  Data\General Data\outlets.csv   120        9               0
          Data\General Data\road_conditions.csv 10920        3               0
        Data\General Data\service_allowance.csv     9        3               0
            Data\General Data\traffic_speed.csv   576        4               0
                 Data\General Data\vehicles.csv    60        9               0
 Data\Submission Templates\submission_task1.csv  5014        3               0
Data\Submission Templates\submission_task2a.csv    60        3               0
Data\Submission Templates\submission_task2b.csv    85        6               0
             Data\Test Data\route_legs_test.csv  5014       18               0
           Data\Test Data\task1_test_inputs.csv  501

## Task 1 label construction

For dispatched orders, `(route_id, seq_in_route)` matches `(route_id, seq)` exactly once. Service begins at `max(actual_arrival, window_open)` so early waiting is excluded. Lateness is a strict comparison: `actual_arrival > window_close`. Actual-time fields are used only here and never enter the model feature matrix.

In [3]:
label_audit = json.loads((WORK / "reports" / "task1_label_audit.json").read_text(encoding="utf-8"))
print("Formula:", label_audit["formula"])
print("Rows:", label_audit["rows"])
print("Early arrivals:", label_audit["early_arrivals"], f"({label_audit['early_arrival_rate']:.2%})")
print("Service median / mean:", label_audit["service_summary"]["50%"], "/", round(label_audit["service_summary"]["mean"], 3))
print("Late rate:", f"{label_audit['late_rate']:.2%}")
print("Arrival exactly at close (non-late):", label_audit["arrival_equal_close_rows"])
assert label_audit["window_cross_midnight_rows"] == 0
assert label_audit["leave_before_arrival_raw_rows"] == 0

Formula: {'service_start': 'max(actual_arrival, window_open)', 'service_min': 'leave_outlet - service_start', 'late': 'actual_arrival > window_close (strict inequality)'}
Rows: 91894
Early arrivals: 4023 (4.38%)
Service median / mean: 15.0 / 19.003
Late rate: 19.58%
Arrival exactly at close (non-late): 279


## Task 1 chronological validation and final GPU models

In [4]:
gpu = json.loads((WORK / "reports" / "task1_gpu_benchmark.json").read_text(encoding="utf-8"))
comparison = pd.DataFrame({
    "model": ["GPU XGBoost depth-6", "Saved CPU booster", "GPU XGBoost loss-guided", "Saved CPU booster"],
    "target": ["service", "service", "lateness", "lateness"],
    "two-fold mean": [
        gpu["scores"]["service"]["xgb_depth6"]["mean_mae"],
        gpu["scores"]["service"]["cpu_existing"]["mean_mae"],
        gpu["scores"]["late"]["xgb_lossguide"]["mean_log_loss"],
        gpu["scores"]["late"]["cpu_existing"]["mean_log_loss"],
    ],
    "metric": ["MAE minutes", "MAE minutes", "log loss", "log loss"],
})
display(comparison)
print("CUDA training confirmed:", gpu["cuda_training_confirmed"])
print("Selected:", gpu["selected_service_model"], "/", gpu["selected_late_model"])

errors = json.loads((WORK / "reports" / "error_analysis.json").read_text(encoding="utf-8"))
display(pd.DataFrame(errors["top_service_features"][:8]).rename(columns={"service_importance_mean": "permutation importance"}))
display(pd.DataFrame(errors["top_late_features"][:8]).rename(columns={"late_importance_mean": "permutation importance"}))

                  model   target  two-fold mean      metric
    GPU XGBoost depth-6  service       4.044108 MAE minutes
      Saved CPU booster  service       4.060437 MAE minutes
GPU XGBoost loss-guided lateness       0.142450    log loss
      Saved CPU booster lateness       0.146201    log loss
CUDA training confirmed: True
Selected: xgb_depth6 / xgb_lossguide
                    feature  permutation importance
                  outlet_id                5.323785
              festival_ramp                1.642673
                order_units                0.959113
                  is_payday                0.274015
                        dow                0.263862
            order_weight_kg                0.161132
            order_volume_m3                0.132805
planned_margin_to_close_min                0.065024
                       feature  permutation importance
   planned_margin_to_close_min                0.195258
                     outlet_id                0.098730


## Task 2A weekly demand forecasting

Training demand combines every order from the training file and Task 1 test inputs, including deferred and not-run orders, and assigns volume to the requested ISO week. Lag and rolling features are shifted. Future calendar fields are known from the supplied calendar. Every backtest recursively predicts the next 10 weeks.

In [5]:
task2a_report = json.loads((WORK / "reports" / "task2a_model_report.json").read_text(encoding="utf-8"))
rows = []
for target, detail in task2a_report["targets"].items():
    for model, score in detail["candidates"].items():
        rows.append({"target": target, "model": model, "mean WAPE": score["mean_wape"], "latest WAPE": score["latest_wape"], "selected": model == detail["selected"]})
display(pd.DataFrame(rows).sort_values(["target", "mean WAPE"]))
print("Forecast horizon:", task2a_report["forecast_start"], "to", task2a_report["forecast_end"])
print("Selected models:", task2a_report["selected"])
assert task2a_report["missing_series_weeks"] == 0

           target                 model  mean WAPE  latest WAPE  selected
chilled_volume_m3                   hgb   0.039695     0.031660      True
chilled_volume_m3                 ridge   0.039925     0.039686     False
chilled_volume_m3               hgb_log   0.041454     0.033332     False
chilled_volume_m3       historical_mean   0.050527     0.040198     False
chilled_volume_m3             rolling13   0.052458     0.040919     False
chilled_volume_m3              rolling4   0.055273     0.053771     False
chilled_volume_m3 seasonal_recent_blend   0.055952     0.040942     False
chilled_volume_m3            seasonal52   0.068607     0.050180     False
  total_volume_m3                   hgb   0.053232     0.046833      True
  total_volume_m3             rolling13   0.060401     0.056001     False
  total_volume_m3 seasonal_recent_blend   0.066932     0.052305     False
  total_volume_m3       historical_mean   0.066946     0.052289     False
  total_volume_m3              rolling

## Task 2B allocation and official validation

In [6]:
allocation = json.loads((WORK / "reports" / "task2b_optimization_report.json").read_text(encoding="utf-8"))
final_audit = json.loads((WORK / "reports" / "final_audit.json").read_text(encoding="utf-8"))
display(pd.DataFrame(allocation["sensitivity"]).T[["served_orders", "deferred_orders", "deferred_yesterday_served", "days_since_last_served_sum", "chilled_orders_served"]])
print("Selected policy:", allocation["selected_policy"])
print("Maximum feasible order count:", allocation["selected_summary"]["maximum_feasible_order_count"])
print("Official validator:", final_audit["task2b_validator_output"])
assert final_audit["task2b_official_validator"]

served_orders deferred_orders deferred_yesterday_served days_since_last_served_sum chilled_orders_served
           77               8                        10                        130                    19
           77               8                        10                        130                    19
           77               8                        10                        130                    19
           79               6                         9                        128                    21
Selected policy: fairness_first
Maximum feasible order count: 79
Official validator: FEASIBILITY: PASSED - every rule satisfied.


## Final saved-model inference demonstration

## Reproduce the modelling pipeline

The cells below call the same local source modules used to create the submitted models and predictions. They document label construction, preprocessing, fitting, chronological evaluation, allocation, and the final audit. The organiser's `Data` folder is required. Set `RUN_FULL_REPRODUCTION = True` to regenerate artifacts in a separate `work` directory; Task 1 retraining also requires a CUDA-capable GPU. The switch is off in this submitted copy so the saved-model inference cell remains quick to run. The training cells have not been re-executed in this package because the raw competition data is excluded.

In [7]:
# Reproduction controls: outputs go to PROJECT/work, leaving the submitted files intact.
RUN_FULL_REPRODUCTION = True
REPRO = PROJECT / "work"
if RUN_FULL_REPRODUCTION:
    for folder in ("features", "models", "submissions", "reports", "experiments", "figures"):
        (REPRO / folder).mkdir(parents=True, exist_ok=True)
print("Full reproduction enabled:", RUN_FULL_REPRODUCTION)

Full reproduction enabled: True


### Label construction and preprocessing

In [8]:
if RUN_FULL_REPRODUCTION:
    from audit_data import main as audit_raw_data
    from labels import main as construct_task1_labels
    from features_task1 import build_task1_feature_frame
    from features_task2a import build_weekly_history

    audit_raw_data()
    construct_task1_labels()  # service starts at max(arrival, window open); late means arrival > close
    task1_train_features = build_task1_feature_frame("train")
    task1_test_features = build_task1_feature_frame("test")
    weekly_demand = build_weekly_history()  # includes deferred and not-run orders
    print("Task 1 train/test feature rows:", len(task1_train_features), len(task1_test_features))
    print("Task 2A depot/brand/week rows:", len(weekly_demand))

{
  "python": "3.12.3 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:42:21) [MSC v.1916 64 bit (AMD64)]",
  "platform": "Windows-11-10.0.26200-SP0",
  "pandas": "3.0.2",
  "numpy": "2.4.4",
  "sklearn": "1.8.0"
}
Data\General Data\calendar.csv: 910 rows x 12 cols
  columns=['date', 'dow', 'dow_name', 'is_weekend', 'iso_year', 'iso_week', 'is_payday', 'festival', 'festival_ramp', 'is_holiday', 'monsoon', 'is_operating']
  date_ranges={'date': {'min': '2024-01-01', 'max': '2026-06-28', 'unparsed_nonmissing': 0}}
  missing={'festival': 892}
Data\General Data\district_travel.csv: 12 rows x 8 cols
  columns=['district', 'depot', 'road_class', 'free_flow_kmh', 'depot_to_district_km', 'depot_to_district_freeflow_min', 'inter_stop_km', 'inter_stop_freeflow_min']
Data\General Data\outlets.csv: 120 rows x 9 cols
  columns=['outlet_id', 'brand', 'district', 'depot', 'dock_type', 'parking_constraint', 'mall_window', 'window_open_time', 'window_close_time']
  missing={'mall_window': 108}
Dat

### Fit and compare Task 1 and Task 2A models

In [9]:
if RUN_FULL_REPRODUCTION:
    import shutil
    from train_task1 import main as train_task1_baselines
    from train_task1_gpu import main as train_task1_final
    from train_task2a import main as train_task2a_final

    train_task1_baselines()  # chronological CPU baselines and evaluation report
    shutil.copy2(REPRO / "models" / "task1_models.pkl", REPRO / "models" / "task1_models_cpu.pkl")
    train_task1_final()      # chronological GPU model comparison and final Task 1 outputs
    train_task2a_final()     # rolling ten-week backtests and final Task 2A outputs

{
  "splits": [
    {
      "train_start": "2024-01-01",
      "train_end": "2025-10-11",
      "valid_start": "2025-10-13",
      "valid_end": "2025-11-22",
      "train_rows": 76980,
      "valid_rows": 4987
    },
    {
      "train_start": "2024-01-01",
      "train_end": "2025-11-22",
      "valid_start": "2025-11-24",
      "valid_end": "2026-01-03",
      "train_rows": 81967,
      "valid_rows": 4892
    },
    {
      "train_start": "2024-01-01",
      "train_end": "2026-01-03",
      "valid_start": "2026-01-05",
      "valid_end": "2026-02-14",
      "train_rows": 86859,
      "valid_rows": 5035
    }
  ],
  "service": {
    "global_median": {
      "folds": [
        {
          "mae": 9.745939442550632,
          "rmse": 17.721827338821193,
          "median_ae": 5.0,
          "bias": -6.4962903549228
        },
        {
          "mae": 9.084423548650859,
          "rmse": 17.382257803434317,
          "median_ae": 5.0,
          "bias": -5.292518397383483
        },
    

### Held-out analysis, peak-day allocation and output validation

In [10]:
if RUN_FULL_REPRODUCTION:
    from evaluate import main as evaluate_holdout
    from optimize_task2b import main as allocate_peak_day
    import inference
    from audit import main as validate_outputs

    allocate_peak_day()     # Task 2B allocation and prioritization comparison
    def validate_with_checker_layout():
        # The supplied checker searches under Data/data, while the modelling code uses Data.
        import os
        checker_data = DATA / "data"
        checker_data.mkdir(exist_ok=True)
        created_links = []
        try:
            for relative in (
                "Test Data/task2b_peak_day_scenarios.csv",
                "Test Data/task2b_peak_day_fleet.csv",
                "General Data/vehicles.csv",
                "General Data/district_travel.csv",
                "General Data/service_allowance.csv",
            ):
                source = DATA / relative
                link = checker_data / source.name
                if not link.exists():
                    os.link(source, link)
                    created_links.append(link)
            validate_outputs()
        finally:
            for link in created_links:
                link.unlink()
            if not any(checker_data.iterdir()):
                checker_data.rmdir()

    evaluate_holdout()      # subgroup errors, diagnostics and feature importance
    original_inference_work = inference.WORK
    inference.WORK = REPRO
    try:
        validate_with_checker_layout()  # template checks, model reload and official Task 2B checker
    finally:
        inference.WORK = original_inference_work
    print("Reproduced artifacts:", REPRO)

fairness_first {'policy': 'fairness_first', 'solver_status': 0, 'solver_message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)', 'objective_minimized': -303609116.1740057, 'served_orders': 77, 'deferred_orders': 8, 'served_volume_m3': 313.829, 'served_weight_kg': 55705.70000000001, 'deferred_yesterday_served': 10, 'deferred_yesterday_total': 10, 'days_since_last_served_sum': 130, 'fresh_orders_served': 68, 'chilled_orders_served': 19, 'van_only_served': 5, 'variables': 210, 'constraints': 272}
cold_chain {'policy': 'cold_chain', 'solver_status': 0, 'solver_message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)', 'objective_minimized': -303799116.1740029, 'served_orders': 77, 'deferred_orders': 8, 'served_volume_m3': 313.829, 'served_weight_kg': 55705.70000000001, 'deferred_yesterday_served': 10, 'deferred_yesterday_total': 10, 'days_since_last_served_sum': 130, 'fresh_orders_served': 68, 'chilled_orders_served': 19, 'van_only_served': 5, 'variabl

In [11]:
# FINAL CELL: load the saved models and run inference without retraining.
from inference import predict_task1, predict_task2a

task1_bundle = joblib.load(WORK / "models" / "task1_models.pkl")
task2a_bundle = joblib.load(WORK / "models" / "task2a_models.pkl")

task1_inputs = pd.read_csv(DATA / "Test Data" / "task1_test_inputs.csv").head(5)
task1_predictions = predict_task1().head(5)
print("TASK 1 REPRESENTATIVE INPUTS")
display(task1_inputs[["delivery_id", "outlet_id", "brand", "order_volume_m3", "planned_arrival_time", "window_close_time"]])
print("TASK 1 PREDICTIONS")
display(task1_predictions)

task2a_inputs = pd.read_csv(DATA / "Test Data" / "task2a_test_inputs.csv").head(6)
task2a_predictions = predict_task2a().head(6)
print("TASK 2A REPRESENTATIVE INPUTS")
display(task2a_inputs)
print("TASK 2A PREDICTIONS")
display(task2a_predictions)

print("Loaded Task 1 models:", task1_bundle["selected_service_model"], "/", task1_bundle["selected_late_model"])
print("Loaded Task 2A models:", task2a_bundle["selected"])
print("Inference completed from saved model files.")

TASK 1 REPRESENTATIVE INPUTS
delivery_id outlet_id brand  order_volume_m3 planned_arrival_time window_close_time
 ORD0092308    OUT001 Fresh            0.501                05:21             07:30
 ORD0092309    OUT002 Fresh            0.539                06:10             08:00
 ORD0092310    OUT003 Fresh            0.472                05:46             07:30
 ORD0092311    OUT004 Fresh            1.193                06:44             08:00
 ORD0092312    OUT005 Fresh            2.687                06:22             07:45
TASK 1 PREDICTIONS
delivery_id  pred_service_min  pred_late_prob
 ORD0092308          8.361660        0.000155
 ORD0092309          8.067531        0.000530
 ORD0092310         12.087658        0.000319
 ORD0092311         11.857336        0.004319
 ORD0092312         14.509268        0.008601
TASK 2A REPRESENTATIVE INPUTS
row_id      depot brand  iso_year  iso_week
 W0000      Kandy Fresh      2026        14
 W0001      Kandy Style      2026        14
 W0002    